# Problem Restatement

## The question (Q5)
Past bond programs were structured to recommend projects in each Ward to try to
maximize the likelihood of a successful outcome in a voter election. Can the results
be provided in a map format overlaid on the City Ward Map to make sure we are
getting good representation for each Ward? This may require that we have to look at
more than the top 20 locations in the City. In mapping these
collisions/volumes/collision-volume aggregate, we would envision some sort of
heat map for demonstration purposes.

## How my analysis will answer it
My analysis will enable the City of Norman to identify which wards need the most attention in terms of road safety. Having a ranked list of dangerous and a heat map can possibly reveal patterns about dangerous road sections.

**Method:** every road segment gets a volume-to-capacity (V/C) ratio and a collision count. The segments are then ranked **inside each of the 8 wards** three ways: by V/C, by collisions, and by the City's aggregate (V/C rank + collision rank). Ranking inside each ward, instead of making one citywide top 20, is what guarantees every ward is represented. The maps and heat map are built from these lists in Checkpoint 3.

# Data Loading & Inspection

In [ ]:
# Mount Google Drive and import libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

# Load ACOG Count Files
shapelist = []
input_years = ['19','21','23','25','26']
df = {}
pd.set_option("display.max_colwidth", None)
for year in input_years:
  dataframe = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"+"ACOG FY" + year + ".xlsx")
  df[year] = dataframe
  shapelist.append(df[year])
  print(f"Read dataframe year: {year}")

# Load Collision workbook
crashesdf = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Traffic_Project_OU_FALL_PROJECT.xlsx", sheet_name='Base_Data')
shapelist.append(crashesdf)
print("Read crash list dataframe")

# Load the Code_Book
codebook = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Code_Book.xlsx")
print("Read Code Book")
shapelist.append(codebook)

# Load the GeoJSON
wards = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Council_Wards_4934068591990916100.geojson')
shapelist.append(wards)
print("Read Ward GeoJSON file")

## Data problems I noticed

1. It's curious to me that every ACOG has exactly 100 entries
2. It's unfortunate that the columns names differ from year to year because that means I'll need to rename them.
3. The ACOGs don't record data in the same format. `df['19']` and `df['21']` have `Latest` and `Latest Date` but the others don't.
4. The ACOG vehicles per day has several typos like `3,3339` and `4, 436`
5. The crashes data needs to have the locations translated in order to properly plot them on a heatmap
6. Some crash coordinates are `-361`
7. FY25 and FY26 have 101 rows, not 100. The extra row in each has no ID and no coordinates. One FY26 row has no count at all: it says "Under widening construction".
8. The traffic count is not a number. It is text like `EB 2,913vpd - 37mph / WB 2,651vpd`, with the two directions, the speeds and the date all in one cell.
9. Collisions: 272 rows repeat an `AccidentNumber` that is already listed (the same crash written twice with two directions).
10. Collisions: `Lat`/`Lon` are the coordinates of the nearest intersection, not of the crash, and about 1,700 crashes are on private property (parking lots), not on a road.
11. Code_Book: the collision file stores causes as ID numbers (like `-67028`), so they mean nothing until they are looked up in the Code_Book's `EntryID` column.
12. Ward GeoJSON: no errors found. It has 8 polygons in plain latitude/longitude, the same kind of numbers as the other files. It has more columns than needed, and a road that runs along a ward boundary belongs to two wards.


In [ ]:
# Shows the shapes of each dataframe
i = 0
for item in shapelist:
  print(f"\033[92mItem {i} shape: \033[0m{item.shape}")
  print(f"\033[92mItem {i} columns: \033[0m{item.columns}")
  print(" ")
  i = i + 1


# Data Preparation & Transformation

In [ ]:
# Rename the columns
dictionary = {
    'Loc ID':  'loc_id',
    'LOCAL_ID': 'loc_id',
    'On':      'on_road',
    'ON_ROAD': 'on_road',
    'From': 'from',
    'FROM': 'from',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'Unnamed: 16': 'count_text',
    'Unnamed: 13': 'count_text',
    'Unnamed: 11': 'count_text',
    'LATEST_COUNT': 'count_text',
    'To': 'to',
    'TO': 'to',
    'Functional Class': 'road_type',
    'Latest': 'latest',
    'Latest Date': 'latest_date'
}
for year in df:
    df[year] = df[year].rename(columns=dictionary)

In [ ]:
# Find VPD_max
#### (A.I. was used for this RegEx) ####
DIR = r'(?:NB|SB|EB|WB)'
NUM = r'(\d[\d,]*(?:\s\d{3})?)'
VPD1_RE = DIR + r'\D*?' + NUM
VPD2_RE = r'mph\s*(?:[/-]\s*' + DIR + r'?|' + DIR + r')\s*' + NUM
for frame in df:
  # Fix a typo in FY19 (one digit too many) before reading the numbers (from Tutorial 2)
  df[frame]['count_text'] = df[frame]['count_text'].str.replace('3,3339vpd', '3,339vpd', regex=False)
  df[frame]['VPD1'] = df[frame]['count_text'].str.extract(VPD1_RE)[0]
  df[frame]['VPD2'] = df[frame]['count_text'].str.extract(VPD2_RE)[0]

# Fix Dates
df['26']['road_type'] = '-'
for year in df:
  df[year]['count_text'] = df[year]['count_text'].str.replace(")", "", regex=False)
  df[year]['count_text'] = df[year]['count_text'].str.replace("(", "", regex=False)
for year in df:
  df[year]['date'] = df[year]['count_text'].str.extract(r"(\d{1,2}[-/]\d{1,2}[-/]\d{2,4})\s*$")[0]
  df[year]['date'] = pd.to_datetime(df[year]['date'], errors="coerce")

# create `latest` if necessary
for frame in df:
  if 'latest' not in df[frame].columns:
    df[frame]['latest'] = 0

# Shrink the dataframes
df_reduced = {}
for frame in df:
  df_reduced[frame] = df[frame][['loc_id','date','on_road','from','latitude','longitude','count_text','to','road_type','VPD1','VPD2', 'latest']].copy()
  df_reduced[frame]['year'] = 'FY' + frame        # remember which file each row came from

# Combine the dataframes
big_frame = pd.concat(df_reduced)

In [ ]:
# Find VPD_max
big_frame['VPD1'] = big_frame['VPD1'].str.replace(",", "", regex=False)
big_frame['VPD2'] = big_frame['VPD2'].str.replace(",", "", regex=False)
big_frame['VPD1'] = big_frame['VPD1'].str.replace(" ", "", regex=False)
big_frame['VPD2'] = big_frame['VPD2'].str.replace(" ", "", regex=False)
big_frame['VPD1'] = pd.to_numeric(big_frame['VPD1'], errors='coerce')
big_frame['VPD2'] = pd.to_numeric(big_frame['VPD2'], errors='coerce')
big_frame[['VPD1','VPD2']] = big_frame[['VPD1','VPD2']].fillna(0)
big_frame['VPD_sum'] = big_frame['VPD1'] + big_frame['VPD2']
big_frame['VPD_max'] = big_frame[['latest','VPD_sum']].max(axis=1)
big_frame.drop('VPD1', axis=1, inplace=True)
big_frame.drop('VPD2', axis=1, inplace=True)
big_frame.drop('VPD_sum', axis=1, inplace=True)

In [ ]:
big_frame.head(25)

In [ ]:


from shapely.geometry import Point
import matplotlib.pyplot as plt
wards = gpd.read_file('/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/Council_Wards_4934068591990916100.geojson')

fig, ax = plt.subplots(figsize=(10, 8))
wards.plot(ax=ax, column='WARDNO', categorical=True, cmap='Pastel1', edgecolor='black', legend=True)

for i in range(len(wards)):
    shape = wards['geometry'].iloc[i]
    center = shape.representative_point()        # a point that is always inside the shape
    ax.text(center.x, center.y, 'Ward ' + str(wards['WARDNO'].iloc[i]), ha='center', fontsize=11, weight='bold')

ax.set_title('City of Norman — City Council Wards')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.show()


In [ ]:
def find_ward(lat, lon):
    """Return the ward number that contains (lat, lon), or None."""
    p = Point(lon, lat)                 # careful: Point takes (x = longitude, y = latitude)
    for i in range(len(wards)):
        if wards['geometry'].iloc[i].contains(p):
            return wards['WARDNO'].iloc[i]
    return None

def ward_for_row(row):
    return find_ward(row['latitude'], row['longitude'])

big_frame['ward'] = big_frame.apply(ward_for_row, axis=1)
big_frame.head(10)

## Road segments of at least one mile
*The next three cells are adapted from Tutorial 2 (Steps 6 and 8).*

1. **Clean the road names.** `W Robinson St`, `E Robinson St` and `E Robinson St E` are one road, so each name is reduced to a key like `ROBINSON`. Numbered avenues keep their side of town (`12TH E` and `12TH W` are different roads).
2. **Cut each road into one-mile segments.** Norman's main roads sit on a one-mile grid. A segment is one road between two neighboring grid roads, for example *Robinson: 36th Ave NW – 24th Ave NW*. An east-west road is cut using the count station's longitude, and a north-south road using its latitude.
3. **Keep the study roads and the highest volume.** The City's jargon sheet says to use arterials and section-line roads, not freeways, collectors or local streets. A segment can hold several count stations and several years, so `groupby` + `max` keeps the highest volume, as the City asked.

In [ ]:
# Clean the road names (adapted from Tutorial 2, Step 6)
DATA_FOLDER = "/content/drive/MyDrive/Colab Notebooks/MIS3213/Norman_Traffic_Analysis/data/"

REMOVE_WORDS = ['N', 'S', 'E', 'W', 'AVE', 'AVENUE', 'ST', 'STREET', 'RD', 'ROAD', 'BLVD', 'BLV', 'DR', 'PKWY',
                'LN', 'LANE', 'HWY', 'CT', 'CIR', 'PL', 'TRL', 'WAY']
QUADRANTS = ['NE', 'NW', 'SE', 'SW']
NAME_ALIASES = {'CEDAR': 'CEDAR LANE', 'STATE 9': 'SH 9'}

def road_key(name):
    """'W Robinson St' -> 'ROBINSON'     '12th Ave NE' -> '12TH E'"""
    if not isinstance(name, str):          # empty cells are not text
        return ''
    name = name.upper()
    for ch in '.,()':
        name = name.replace(ch, ' ')
    side = ''
    kept = []
    for word in name.split():
        if word in QUADRANTS:
            side = word[1]                 # 'NE' -> 'E'
        elif word not in REMOVE_WORDS:
            kept.append(word)
    key = ' '.join(kept)
    key = NAME_ALIASES.get(key, key)
    if key[:1].isdigit():                  # numbered avenue: add the side of town
        key = key + ' ' + side
    return key

big_frame['road'] = big_frame['on_road'].apply(road_key)

# Three station names in FY23 are garbled, e.g. '(E Lindsey St)12th Ave NW'.
# The station ID never changes, so the other years show the real road.
NAME_FIXES_BY_ID = {'52500-135': 'LINDSEY', '52500-16': '48TH W', '52500-293': 'INDIAN HILLS'}
for station_id, right_name in NAME_FIXES_BY_ID.items():
    big_frame.loc[(big_frame['loc_id'] == station_id) & (big_frame['year'] == 'FY23'), 'road'] = right_name

big_frame[['on_road', 'road']].drop_duplicates().head(12)

In [ ]:
# Cut the roads into one-mile segments (adapted from Tutorial 2, Step 8)
NS_GRID = {   # north-south grid roads: longitude -> name
    -97.5473: '72nd Ave NW/SW', -97.5296: '60th Ave NW/SW', -97.5120: '48th Ave NW/SW',
    -97.4943: '36th Ave NW/SW', -97.4767: '24th Ave NW/SW', -97.4590: '12th Ave NW / Berry Rd',
    -97.4412: 'Porter Ave / Jenkins Ave', -97.4236: '12th Ave NE/SE', -97.4059: '24th Ave NE/SE',
    -97.3882: '36th Ave NE/SE', -97.3706: '48th Ave NE/SE', -97.3530: '60th Ave NE/SE',
    -97.3354: '72nd Ave NE/SE', -97.3180: '84th Ave NE/SE', -97.3004: '96th Ave NE/SE',
    -97.2828: '108th Ave NE/SE', -97.2652: '120th Ave NE/SE', -97.2476: '132nd Ave NE/SE',
    -97.2300: '144th Ave NE/SE', -97.2123: '156th Ave NE/SE', -97.1945: '168th Ave NE/SE', -97.1770: '180th Ave NE/SE'}
EW_GRID = {   # east-west grid roads: latitude -> name
    35.3340: 'County line', 35.3195: 'Stella Rd', 35.3051: 'Bethel Rd', 35.2908: 'Indian Hills Rd',
    35.2763: 'Franklin Rd', 35.2619: 'Tecumseh Rd', 35.2474: 'Rock Creek Rd', 35.2329: 'Robinson St',
    35.2184: 'Main St / Alameda St', 35.2039: 'Lindsey St', 35.1894: 'Imhoff Rd', 35.1751: 'Cedar Lane Rd',
    35.1604: 'Post Oak Rd', 35.1458: 'Etowah Rd', 35.1313: 'Slaughterville Rd'}

NORTH_SOUTH_ROADS = ['BERRY', 'PORTER', 'JENKINS', 'FLOOD', 'CLASSEN', 'CHAUTAUQUA', 'UNIVERSITY',
                     'WEBSTER', 'JAMES GARNER', 'ED NOBLE', 'BROADWAY']

def is_north_south(road):
    return road in NORTH_SOUTH_ROADS or road[:1].isdigit()

def find_between(value, grid):
    """Return the two grid positions on either side of `value`, or (None, None)."""
    positions = sorted(grid.keys())
    for i in range(len(positions) - 1):
        if positions[i] <= value < positions[i + 1]:
            return positions[i], positions[i + 1]
    return None, None

def road_title(road):
    if road[:1].isdigit():
        number, side = road.split()
        return number.lower() + ' Ave ' + ('NE/SE' if side == 'E' else 'NW/SW')
    return road.title()

def segment_name(road, lat, lon):
    """Name of the one-mile segment of `road` that contains the point (lat, lon)."""
    if is_north_south(road):
        grid = EW_GRID                     # a north-south road is cut by the east-west grid roads
        low, high = find_between(lat, grid)
    else:
        grid = NS_GRID
        low, high = find_between(lon, grid)
    if low is None:
        return None
    return road_title(road) + ': ' + grid[low] + ' – ' + grid[high]

def segment_for_count(row):
    return segment_name(row['road'], row['latitude'], row['longitude'])

big_frame['segment'] = big_frame.apply(segment_for_count, axis=1)
big_frame[['year', 'on_road', 'from', 'to', 'segment']].head(10)

In [ ]:
# Keep only the roads in the study, then one row per segment with the highest volume
# (adapted from Tutorial 2, Step 8)
SECTION_LINE_ROADS = ['ROBINSON', 'MAIN', 'ALAMEDA', 'LINDSEY',
                      'IMHOFF', 'CEDAR LANE', 'POST OAK', 'ETOWAH',
                      'TECUMSEH', 'ROCK CREEK', 'FRANKLIN', 'INDIAN HILLS', 'BETHEL', 'STELLA',
                      'BERRY', 'PORTER', 'JENKINS']
FREEWAYS = ['SH 9', 'I 35', 'INTERSTATE']
MANUAL_EXCLUDE = ['DAWS']        # ACOG says Minor Arterial, but it is a ~800 vpd campus street

arterial_roads = big_frame[big_frame['road_type'].isin(['Principal Arterial', 'Minor Arterial'])]['road'].unique()

def in_study(road):
    if road in FREEWAYS or road in MANUAL_EXCLUDE:
        return False
    return road in SECTION_LINE_ROADS or road[:1].isdigit() or road in arterial_roads

big_frame['in_study'] = big_frame['road'].apply(in_study)

print('Count records:', len(big_frame))
print('  no count that year (VPD_max is 0):', (big_frame['VPD_max'] == 0).sum())
print('  no coordinates, so no segment:', big_frame['segment'].isna().sum())
print('  on a road left out of the study:', (~big_frame['in_study']).sum())
print('Roads left out:', sorted(big_frame[~big_frame['in_study']]['road'].unique()))

study = big_frame[big_frame['in_study'] & (big_frame['VPD_max'] > 0) & big_frame['segment'].notna()]

# A segment can have several stations and several years: the highest volume wins
segments = study.groupby('segment').agg(road=('road', 'first'),
                                        volume=('VPD_max', 'max'),
                                        stations=('loc_id', 'nunique'),
                                        latitude=('latitude', 'mean'),
                                        longitude=('longitude', 'mean')).reset_index()
segments['volume'] = segments['volume'].astype(int)
print('Segments:', len(segments))
segments.sort_values('volume', ascending=False).head(10)

## Place each collision on the road where it happened
*The first cell is adapted from Tutorial 1 (Part B).*

* Repeated `AccidentNumber`s are counted once.
* Private-property collisions and collisions with no coordinates (`-361`) are left out.
* The coordinates given are the **intersection**, so a "From" collision is moved the stated number of feet north, south, east or west.

Then each collision is given a segment the same way the traffic counts were: its road is `StreetName` (the road the collision was **on**, not the cross street), and its moved location picks the mile. A collision coded at Franklin Rd & 48th Ave NE that happened on 48th Ave belongs to 48th Ave. Collisions that do not land on a segment with a traffic count go no further, as the City's instructions say.

In [ ]:
# Put each collision at its real location (adapted from Tutorial 1, Part B)
import numpy as np
import math

crashes = crashesdf.copy()
print('Collision records:', len(crashes))

# 1. Some collisions are listed twice with the same AccidentNumber. Keep each one once.
crashes = crashes.drop_duplicates('AccidentNumber')
print('Unique collisions:', len(crashes))

# 2. Private property (parking lots) is not a public road
crashes['private'] = crashes['StreetName'].str.contains('PRIVATE', na=False)

# 3. -361 is the City's code for "no coordinates"
crashes.loc[crashes['Lat'] < -90, 'Lat'] = np.nan
crashes.loc[crashes['Lon'] < -180, 'Lon'] = np.nan
print('  on private property:', crashes['private'].sum())
print('  public road but no coordinates:', (crashes['Lat'].isna() & ~crashes['private']).sum())

crashes = crashes[~crashes['private'] & crashes['Lat'].notna()].copy()
print('Collisions we can locate:', len(crashes))

# 4. Lat/Lon are the INTERSECTION. Move "From" collisions the stated distance and direction.
FEET_PER_DEGREE_LAT = 364000

def move_point(lat, lon, direction, distance_ft):
    """Move a point `distance_ft` feet in `direction`. Returns (new_lat, new_lon)."""
    if pd.isna(distance_ft) or distance_ft > 5280:        # unknown or placeholder (9999) distance: don't move
        return (lat, lon)
    feet_per_degree_lon = FEET_PER_DEGREE_LAT * math.cos(math.radians(lat))
    if direction == 'DirectionNorth':
        lat = lat + distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionSouth':
        lat = lat - distance_ft / FEET_PER_DEGREE_LAT
    elif direction == 'DirectionEast':
        lon = lon + distance_ft / feet_per_degree_lon
    elif direction == 'DirectionWest':
        lon = lon - distance_ft / feet_per_degree_lon
    return (lat, lon)

def crash_location(row):
    if row['From_At'] == 'From':
        return move_point(row['Lat'], row['Lon'], row['Direction'], row['Distance'])
    return (row['Lat'], row['Lon'])

locations = crashes.apply(crash_location, axis=1)
crashes['crash_lat'] = [pair[0] for pair in locations]
crashes['crash_lon'] = [pair[1] for pair in locations]

crashes[['StreetName', 'StreetSuffix', 'XStreetName', 'From_At', 'Direction', 'Distance',
         'Lat', 'Lon', 'crash_lat', 'crash_lon']].head(8)

In [ ]:
# Find the segment each collision belongs to

# The road the collision happened ON is StreetName (+ NE/NW/SE/SW from StreetSuffix).
# road_key() turns it into the same road names the traffic counts use.
crashes['street'] = crashes['StreetName'] + ' ' + crashes['StreetSuffix'].fillna('')
crashes['road'] = crashes['street'].apply(road_key)

# Only collisions on a road that has traffic counts go further (the City's hint)
on_counted_road = crashes['road'].isin(segments['road'])
print('Collisions on a road with traffic counts:', on_counted_road.sum(), 'of', len(crashes))
counted = crashes[on_counted_road].copy()

# Same grid as the traffic counts, but using the collision's moved location
def segment_for_crash(row):
    return segment_name(row['road'], row['crash_lat'], row['crash_lon'])

counted['segment'] = counted.apply(segment_for_crash, axis=1)

# Keep only collisions that land on a mile of road where traffic was actually counted
counted = counted[counted['segment'].isin(segments['segment'])]
print('Collisions on a counted segment:', len(counted))

counted[['StreetName', 'StreetSuffix', 'XStreetName', 'From_At', 'Direction', 'Distance', 'segment']].head(8)

# Metric Computation

## V/C ratio
*Adapted from Tutorial 2 (Steps 9 and 10).*

The number of lanes for each segment comes from `lanes_lookup.csv` (codes like `2`, `4U` = four lanes undivided, `4D` = four lanes divided, `5` = four lanes plus a center turn lane). The lane code is looked up in the ACOG capacity table, and **V/C = volume ÷ capacity**. A V/C of 1.0 means the road carries exactly what it was built for.

The **LOS E** capacity column is used because the jargon sheet defines capacity as the upper limit of traffic before congestion, which is what LOS E measures. It is also the newer (2025) plan.

In [ ]:
# Lanes, capacity and V/C (adapted from Tutorial 2, Steps 9 and 10)
lanes = pd.read_csv(DATA_FOLDER + 'lanes_lookup.csv', dtype=str)

segments = pd.merge(segments, lanes[['segment', 'lanes']], on='segment', how='left', indicator=True)
print(segments['_merge'].value_counts())
print('Segments without lanes:', list(segments[segments['_merge'] == 'left_only']['segment']))

segments['lanes'] = segments['lanes'].fillna('2')       # assume 2 lanes if unknown
segments = segments.drop(columns='_merge')

# ACOG capacity table, LOS E column (2025 OCARTS plan), in vehicles per day
CAPACITY_LOS_E = {'2': 17100, '3': 17100, '4U': 34200, '4D': 38000, '5': 36000,
                  '6U': 52300, '6D': 58000, '1W2': 22000, '1W3': 33000}

segments['capacity'] = segments['lanes'].map(CAPACITY_LOS_E)
segments['vc'] = (segments['volume'] / segments['capacity']).round(3)

segments[['segment', 'volume', 'lanes', 'capacity', 'vc']].sort_values('vc', ascending=False).head(10)

## Collisions per segment

In [ ]:
# Count the collisions on each segment
collision_counts = counted.groupby('segment').size().reset_index(name='collisions')

segments = pd.merge(segments, collision_counts, on='segment', how='left')
segments['collisions'] = segments['collisions'].fillna(0).astype(int)     # no match = 0 collisions

print('Segments with at least one collision:', (segments['collisions'] > 0).sum(), 'of', len(segments))
segments[['segment', 'volume', 'vc', 'collisions']].sort_values('collisions', ascending=False).head(10)

## Wards
Each segment is rebuilt as a line between its two grid roads, and every ward within about 300 feet of that line is recorded. A road that runs **along** a ward boundary is therefore counted in **both** wards, as the City asked. The first and last 10% of the line are skipped so that a segment is not put in a ward it only touches at the intersection at its end.

In [ ]:
# Which ward(s) is each segment in?
from shapely.geometry import LineString

NEAR = 0.001      # about 300 feet, in degrees

def segment_wards(row):
    """Return a list of every ward the segment runs through or along."""
    # Rebuild the segment as a line between its two grid roads
    if is_north_south(row['road']):
        low, high = find_between(row['latitude'], EW_GRID)
        trim = (high - low) * 0.1                      # skip the intersections at both ends
        line = LineString([(row['longitude'], low + trim), (row['longitude'], high - trim)])
    else:
        low, high = find_between(row['longitude'], NS_GRID)
        trim = (high - low) * 0.1
        line = LineString([(low + trim, row['latitude']), (high - trim, row['latitude'])])

    found = []
    for i in range(len(wards)):
        if wards['geometry'].iloc[i].distance(line) < NEAR:
            found.append(int(wards['WARDNO'].iloc[i]))
    return found

segments['wards'] = segments.apply(segment_wards, axis=1)
segments['number_of_wards'] = segments['wards'].apply(len)
print(segments['number_of_wards'].value_counts().sort_index())

# One row per (segment, ward): a segment in two wards appears in both wards' lists
by_ward = segments.explode('wards').rename(columns={'wards': 'ward'})
by_ward = by_ward[by_ward['ward'].notna()].copy()
by_ward['ward'] = by_ward['ward'].astype(int)
print('Rows in the ward table:', len(by_ward))
by_ward.groupby('ward').size()

## Rankings
Three rankings, each made separately inside every ward (`groupby('ward')`), so there are 8 lists of each kind:

1. **V/C rank**: highest V/C is 1
2. **Collision rank**: most collisions is 1
3. **Aggregate**: for segments that have both a volume and collisions, add the two rank numbers. The smallest sum is the top priority. (City's example: 2nd in collisions + 4th in V/C = 6.)

Ties share the same rank (`method='min'`).

In [ ]:
# The three rankings, separately inside each ward

# 1. V/C: highest V/C = rank 1
by_ward['vc_rank'] = by_ward.groupby('ward')['vc'].rank(ascending=False, method='min').astype(int)

# 2. Collisions: most collisions = rank 1
by_ward['collision_rank'] = by_ward.groupby('ward')['collisions'].rank(ascending=False, method='min').astype(int)

# 3. Aggregate: only segments with BOTH volume and collisions.
#    Add the two rank numbers; the smallest sum is the top priority.
both = by_ward[by_ward['collisions'] > 0].copy()
both['vc_rank_both'] = both.groupby('ward')['vc'].rank(ascending=False, method='min')
both['collision_rank_both'] = both.groupby('ward')['collisions'].rank(ascending=False, method='min')
both['rank_sum'] = (both['vc_rank_both'] + both['collision_rank_both']).astype(int)
both['priority_rank'] = both.groupby('ward')['rank_sum'].rank(method='min').astype(int)

by_ward = pd.merge(by_ward, both[['segment', 'ward', 'rank_sum', 'priority_rank']], on=['segment', 'ward'], how='left')

# The same aggregate for the whole city, to compare with the ward lists later
city = segments[segments['collisions'] > 0].copy()
city['city_rank_sum'] = city['vc'].rank(ascending=False, method='min') + city['collisions'].rank(ascending=False, method='min')
city['city_rank'] = city['city_rank_sum'].rank(method='min').astype(int)

print('Rows ranked in the aggregate lists:', by_ward['priority_rank'].notna().sum(), 'of', len(by_ward))
by_ward.sort_values(['ward', 'priority_rank']).head(10)

# Business Question

**Question (Q5):** can the results be shown by ward, so that every ward is well represented, even if that means looking beyond the top 20 locations in the city?

**Answer: yes, and it is necessary.** The lists below rank the segments inside each ward, so every ward has its own number 1, 2, 3 and so on. The table after the lists shows why a single citywide top 20 would not work.

In [ ]:
# The ranked lists for each ward (top 5 shown; the full lists are saved below)
for w in range(1, 9):
    this_ward = by_ward[by_ward['ward'] == w]
    print()
    print('=============== WARD', w, '(', len(this_ward), 'segments ) ===============')
    print('--- Aggregate priority (V/C rank + collision rank) ---')
    print(this_ward.sort_values('priority_rank')[['priority_rank', 'segment', 'vc', 'collisions', 'rank_sum']].head(5).to_string(index=False))
    print('--- By V/C ---')
    print(this_ward.sort_values('vc_rank')[['vc_rank', 'segment', 'volume', 'lanes', 'vc']].head(5).to_string(index=False))
    print('--- By collisions ---')
    print(this_ward.sort_values('collision_rank')[['collision_rank', 'segment', 'collisions']].head(5).to_string(index=False))

### Why one citywide top 20 is not enough
The table below counts how many of the citywide top 20 segments (by the same aggregate) fall in each ward.

In [ ]:
# Would a single citywide top 20 represent every ward?
top20 = city.sort_values('city_rank').head(20)
top20_by_ward = top20.explode('wards').groupby('wards').size()

representation = pd.DataFrame({'segments in the ward': by_ward.groupby('ward').size(),
                               'in the citywide top 20': top20_by_ward}).fillna(0).astype(int)
representation.index.name = 'ward'
representation

**What the results show**

* **Ward 5 has no segment in the citywide top 20**, even though it has more counted segments than any other ward (33). It is the large rural east side: low volumes and few collisions. A citywide list would give Ward 5 voters nothing. Its own list still has clear leaders: *24th Ave SE: Imhoff Rd – Lindsey St* and *Classen: Post Oak Rd – Cedar Lane Rd*.
* The citywide top 20 is concentrated in the center of town: Wards 2, 4 and 8 each have 6 of the 20 (a segment on a boundary counts for both wards).
* The same few roads are at or near the top of several wards' lists, mostly because they run along ward boundaries: **Robinson St**, **12th Ave NE/SE**, **Classen Blvd** and **Main St**. For example, *Robinson: 36th Ave NW – 24th Ave NW* is number 1 in Wards 2 and 3 and tied for number 1 in Ward 8. One project there would serve three wards.
* The highest V/C in the city is *12th Ave NE: Rock Creek Rd – Tecumseh Rd* (0.97, a two-lane road), but it has fewer collisions, so it is number 2 in Ward 6 instead of number 1. No segment is over capacity (V/C above 1.0).

**Key choices and why**

| Choice | Why |
|---|---|
| Volume = the higher of `Latest` and the two directions added together, then the highest across stations and years in a segment | The City's rule: use the highest volume available |
| One-mile segments from the road grid | The City wants segments of at least a mile; the grid gives the same segment for traffic counts and collisions |
| Arterials and section-line roads only | The jargon sheet excludes freeways, collectors and local streets |
| LOS E capacity | It matches the definition of capacity (the limit before congestion) and is the newer plan |
| Collisions assigned by `StreetName` and the moved location | The collision belongs to the road it was on, not the cross street |
| A boundary segment counts in both wards | The City's instruction |
| Rank-sum aggregate inside each ward | The City's method, applied per ward to answer Q5 |

**Limits of this analysis**

* The lane counts in `lanes_lookup.csv` are estimates that still need to be checked on the City's aerial map. Lanes change V/C a lot.
* 611 collisions on public roads have no coordinates and are left out. Tutorial 1 shows a way to recover some of them from other collisions at the same intersection, which is not done here.
* A collision exactly at a grid intersection is put in the segment on one side of it.
* Collisions cover 2016 to August 2026 and are simple counts, not adjusted for how much traffic a road carries.

In [ ]:
# Save the full lists: one Excel sheet per ward, and one table for the maps in Checkpoint 3
columns = ['ward', 'segment', 'volume', 'lanes', 'capacity', 'vc', 'vc_rank',
           'collisions', 'collision_rank', 'rank_sum', 'priority_rank']

with pd.ExcelWriter(DATA_FOLDER + 'Group101_ward_rankings.xlsx') as writer:
    for w in range(1, 9):
        this_ward = by_ward[by_ward['ward'] == w].sort_values(['priority_rank', 'vc_rank'])
        this_ward[columns].to_excel(writer, sheet_name='Ward ' + str(w), index=False)

by_ward.to_csv(DATA_FOLDER + 'Group101_segments_by_ward.csv', index=False)
print('Saved Group101_ward_rankings.xlsx and Group101_segments_by_ward.csv')

# AI Tutor Use
I used Claude (an AI assistant) throughout this checkpoint, and for more than debugging.

**What the AI wrote**
* The regular expressions that pull the two directional counts and the date out of the count text.
* All of the code from "Road segments of at least one mile" to the end: road names, segments, collision locations, matching collisions to segments, lanes and V/C, wards, rankings, and saving the files. Most of it is adapted from Tutorials 1 and 2, and those cells say so.
* The written explanations in those sections, the added inspection notes (7–12), and the Business Question write-up.

**What I wrote, with debugging help**
* Loading the files, the inspection loop, the rename dictionary, combining the five years, and `VPD_max`.

**Questions I asked and what I changed**
1. *Why were my VPD columns all `None`?* I had used `fillna(0, inplace=True)` and also assigned the result back. `inplace=True` returns `None`, so I removed it.
2. *How do I keep the higher of `Latest` and the summed directions?* An `if` cannot compare two whole columns, so I changed it to `max(axis=1)`.
3. *Why did `find_ward` fail on my table?* It takes one point, and I was passing whole columns. I changed it to run once per row with `.apply(..., axis=1)`.